# A1 · Magazzino frigorifero: dai segnali del PLC alla risposta al cliente (C#)

Stessi numeri nel notebook [PySharp](A1-magazzino-frigorifero-pysharp.ipynb) (facciata `Interop`); un test verifica che coincidano. Kernel JupyterNet su .NET 11.

## Il caso

Un **magazzino frigorifero** a +2 °C custodisce merce di un cliente (una società alimentare). Il PLC dell'impianto invia a uno storico, ogni minuto, la temperatura della cella, lo stato di porta e sbrinamento, la corrente del compressore, la pressione di aspirazione e la vibrazione. Un contatore invia i kWh ogni ora.

**Chi lavora sui dati.** Il *responsabile qualità* deve poter certificare che la catena del freddo non si è rotta (soglia HACCP +4 °C). Il *manutentore* vuole sapere in anticipo se il compressore sta cedendo. Il *responsabile di stabilimento* deve stimare i consumi di energia della prossima settimana. Il *cliente* chiede: **«la mia merce è stata sempre sotto +4 °C?»**

**I dati sono sintetici**, generati da `generate-data.cs` con un seme: servono a verificare i risultati contro la verità nota (`data/verita.json`), cosa che con dati reali non è mai possibile.

| Passo | Domanda | Strumento |
|---|---|---|
| 1 | I dati sono completi? | qualità dei dati |
| 2 | Qual è la temperatura *vera* quando il sensore tace? | filtro di Kalman con `NaN` ([cap. 6](../../../../refs/timeseries/00-formule-e-libreria/06-kalman.md)) |
| 3 | Quanta energia consumeremo? | seasonal naive, Holt-Winters, MASE ([cap. 3](../../../../refs/timeseries/00-formule-e-libreria/03-riferimenti-e-metriche.md), [5](../../../../refs/timeseries/00-formule-e-libreria/05-smoothing.md)) |
| 4 | Il compressore sta cedendo? | GARCH sulla vibrazione ([cap. 8](../../../../refs/timeseries/00-formule-e-libreria/08-garch.md)) |
| 5 | Come si influenzano temperatura, corrente e pressione? | VAR ([cap. 7](../../../../refs/timeseries/00-formule-e-libreria/07-var.md)) |
| 6 | KPI, dashboard e risposta al cliente | tile, grafici, testo con i numeri |

In [ ]:
#load "../../../shared/MathemaSharp.PlotSharp.csx"
using MathemaSharp.TimeSeries;
using System.Globalization;
using System.IO;
using System.Runtime.CompilerServices;

static void P(string label, IEnumerable<double> values) =>
    Console.WriteLine($"{label}: {string.Join(" ", values.Select(v => v.ToString("F6", CultureInfo.InvariantCulture)))}");
static void P(string label, double value) => P(label, new[] { value });
static string Here([CallerFilePath] string path = "") => Path.GetDirectoryName(path)!;   // cartella di questo notebook
static double Num(string s) => s.Length == 0 ? double.NaN : double.Parse(s, CultureInfo.InvariantCulture);
// Valore numerico di una chiave in un JSON piatto (verita.json): bastano due righe, nessuna dipendenza.
static double JsonNumber(string json, string key)
{
    var quote = '"';                                   // niente barre rovesciate: questo codice passa da una stringa Python
    var i = json.IndexOf(quote + key + quote + ":") + key.Length + 3;
    var j = json.IndexOfAny(new[] { ',', (char)10, (char)13 }, i);
    return double.Parse(json[i..j], CultureInfo.InvariantCulture);
}

var data = Path.GetFullPath(Path.Combine(Here(), "../../../../scenari/A-magazzino-frigorifero/data"));
var cella = File.ReadLines(Path.Combine(data, "cella_1min.csv")).Skip(1).Select(l => l.Split(',')).ToArray();
double[] Column(int i) => cella.Select(r => Num(r[i])).ToArray();
double[] temp = Column(1), current = Column(5), pressure = Column(6), vib = Column(7);
const double limit = 4.0;   // soglia HACCP, °C

## 1 · Acquisire: i dati sono completi?
Un buco di comunicazione non è un dato a zero: è un dato **assente**. Prima di ogni calcolo si misura quanto manca e dove.

In [ ]:
var gaps = new List<(int Start, int Length)>();
for (var i = 0; i < temp.Length; i++)
{
    if (!double.IsNaN(temp[i])) continue;
    var j = i;
    while (j < temp.Length && double.IsNaN(temp[j])) j++;
    gaps.Add((i, j - i));
    i = j;
}
P("campioni", temp.Length);
P("buchi_n", gaps.Count);
P("mancanti_min", gaps.Sum(g => g.Length));
P("buco_max_min", gaps.Max(g => g.Length));

// Un buco è "sospetto" se la temperatura ai suoi bordi differisce di più di 1 °C: durante il buco è successo qualcosa.
var suspect = gaps.Where(g => g.Start > 0 && g.Start + g.Length < temp.Length
                              && Math.Abs(temp[g.Start + g.Length] - temp[g.Start - 1]) > 1.0).ToArray();
P("buchi_sospetti_min", suspect.Sum(g => g.Length));

campioni: 4320.000000
buchi_n: 6.000000
mancanti_min: 70.000000
buco_max_min: 20.000000
buchi_sospetti_min: 12.000000

## 2 · Capire la temperatura: il filtro di Kalman
Il sensore ha un rumore noto (σ = 0.08 °C, dal datasheet). Il modello *local level* tratta la temperatura vera come uno stato nascosto che si muove di poco a ogni minuto (varianza $q$); i `NaN` saltano la correzione. La varianza $q$ si sceglie massimizzando la **log-verosimiglianza** su una griglia.

In [ ]:
const double sensorVariance = 0.08 * 0.08;
var qGrid = new[] { 0.001, 0.003, 0.01, 0.03, 0.1 };
var logLik = qGrid.Select(q => KalmanFilter.LocalLevel<double>(q, sensorVariance, temp[0], 1.0).Filter(temp).LogLikelihood).ToArray();
P("kalman_loglik_grid", logLik);
var qBest = qGrid[Array.IndexOf(logLik, logLik.Max())];
P("kalman_q", qBest);

var level = KalmanFilter.LocalLevel<double>(qBest, sensorVariance, temp[0], 1.0).Filter(temp).FilteredFirstState;
var trend = KalmanFilter.LocalLinearTrend<double>(qBest, qBest / 10, sensorVariance, temp[0], 0.0, 1.0).Filter(temp).FilteredFirstState;

kalman_loglik_grid: -646.218827 2049.143120 2688.980428 2051.162914 435.282661
kalman_q: 0.010000

### Il KPI HACCP: quanti minuti sopra +4 °C?
Quattro risposte allo stesso quesito. **La salita avviene dentro un buco di comunicazione**, quindi nessuno strumento la vede per intero: l'unica risposta onesta è un **intervallo**.

In [ ]:
int Above(double[] x) => x.Count(v => v > limit);
int Runs(double[] x) { var n = 0; for (var i = 0; i < x.Length; i++) if (x[i] > limit && (i == 0 || x[i - 1] <= limit)) n++; return n; }

var observed = temp.Count(v => !double.IsNaN(v) && v > limit);
P("haccp_osservati_min", observed);
P("haccp_kalman_livello_min", Above(level));
P("haccp_kalman_trend_min", Above(trend));
P("haccp_escursioni_kalman", Runs(level));
P("temp_max_filtrata", level.Max());
P("haccp_intervallo_min", new[] { (double)observed, observed + suspect.Sum(g => g.Length) });

// Verità: la conosce solo il generatore dei dati sintetici, non uno storico reale.
var truth = File.ReadAllText(Path.Combine(data, "verita.json"));
var truthMinutes = (int)JsonNumber(truth, "minuti_veri_sopra_soglia");
P("haccp_verita_min", truthMinutes);
P("haccp_verita_nell_intervallo", truthMinutes >= observed && truthMinutes <= observed + suspect.Sum(g => g.Length) ? 1 : 0);

haccp_osservati_min: 7.000000
haccp_kalman_livello_min: 8.000000
haccp_kalman_trend_min: 9.000000
haccp_escursioni_kalman: 1.000000
temp_max_filtrata: 4.494347
haccp_intervallo_min: 7.000000 19.000000
haccp_verita_min: 11.000000
haccp_verita_nell_intervallo: 1.000000

In [ ]:
// L'evento: martedì 15 settembre, dalle 13:50 alle 14:40 (minuto 1440 + 830 … 1440 + 880).
int a = 1440 + 13 * 60 + 50, b = 1440 + 14 * 60 + 40;
Chart.Lines("Temperatura durante l'evento (asse x: minuti dalle 13:50)",
    new Line("misura (buco = vuoto)", temp[a..b]),
    new Line("Kalman livello", level[a..b]),
    new Line("Kalman trend", trend[a..b]),
    new Line("soglia +4 °C", Enumerable.Repeat(limit, b - a).ToArray(), Dashed: true));

Temperatura durante l'evento (asse x: minuti dalle 13:50) 1 2 3 4 5 0 10 20 30 40 50 <path d='M64 154.92L76.64 163.92L89.28 171.48L101.92 172.56L114.56 175.8L127.2 174L139.84 177.96L152.48 177.6L165.12 184.08L177.76 183.36L190.4 183.36L203.04 179.76L215.68 170.76L228.32 172.2L240.96 167.88L253.6 158.16L266.24 158.52L278.88 149.88M443.2 85.08L455.84 83.28L468.48 84L481.12 91.92L493.76 95.88L506.4 91.2L519.04 102L531.68 101.64L544.32 104.88L556.96 105.6L569.6 108.12L582.24 110.64L594.88 109.92L607.52 114.6L620.16 121.44L632.8 124.68L645.44 126.12L658.08 124.32L670.72 131.88L683.36 130.8' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 155.12L76.64 161.22L89.28 168.33L101.92 171.26L114.56 174.41L127.2 174.12L139.84 176.78L152.48 177.35L165.12 182.01L177.76 182.95L190.4 183.23L203.04 180.83L215.68 173.85L228.32 172.71L240.96 169.36L253.6 161.6L266.24 159.47L278.88 152.82L291.52 152.82L304.16 152.82L316.8 152.82L329.44 152.82L342.08 152.82L354.72 152.82L367.36 152.82L380 152.82L392.64 152.82L405.28 152.82L417.92 152.82L430.56 152.82L443.2 88.16L455.84 84.67L468.48 84.2L481.12 89.55L493.76 93.94L506.4 92.04L519.04 98.94L531.68 100.81L544.32 103.63L556.96 104.99L569.6 107.16L582.24 109.57L594.88 109.81L607.52 113.13L620.16 118.89L632.8 122.9L645.44 125.13L658.08 124.57L670.72 129.63L683.36 130.44' fill='none' stroke='#eb6834' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 154.39L76.64 161.36L89.28 169.27L101.92 172.34L114.56 175.58L127.2 174.99L139.84 177.71L152.48 178.11L165.12 183.09L177.76 183.87L190.4 183.95L203.04 181.09L215.68 173.25L228.32 172.08L240.96 168.51L253.6 160.08L266.24 158.06L278.88 151.03L291.52 146.89L304.16 142.75L316.8 138.62L329.44 134.48L342.08 130.35L354.72 126.21L367.36 122.07L380 117.94L392.64 113.8L405.28 109.66L417.92 105.53L430.56 101.39L443.2 85.13L455.84 82.55L468.48 82.7L481.12 89.19L493.76 94.27L506.4 92.11L519.04 99.78L531.68 101.68L544.32 104.61L556.96 105.89L569.6 108.07L582.24 110.52L594.88 110.55L607.52 114.04L620.16 120.22L632.8 124.36L645.44 126.49L658.08 125.52L670.72 130.89L683.36 131.48' fill='none' stroke='#1baf7a' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 102L76.64 102L89.28 102L101.92 102L114.56 102L127.2 102L139.84 102L152.48 102L165.12 102L177.76 102L190.4 102L203.04 102L215.68 102L228.32 102L240.96 102L253.6 102L266.24 102L278.88 102L291.52 102L304.16 102L316.8 102L329.44 102L342.08 102L354.72 102L367.36 102L380 102L392.64 102L405.28 102L417.92 102L430.56 102L443.2 102L455.84 102L468.48 102L481.12 102L493.76 102L506.4 102L519.04 102L531.68 102L544.32 102L556.96 102L569.6 102L582.24 102L594.88 102L607.52 102L620.16 102L632.8 102L645.44 102L658.08 102L670.72 102L683.36 102' fill='none' stroke='#eda100' stroke-width='2' stroke-dasharray='6 4' stroke-linejoin='round' stroke-linecap='round'/> misura (buco = vuoto) · x=0: 2.53 misura (buco = vuoto) · x=1: 2.28 misura (buco = vuoto) · x=2: 2.07 misura (buco = vuoto) · x=3: 2.04 misura (buco = vuoto) · x=4: 1.95 misura (buco = vuoto) · x=5: 2 misura (buco = vuoto) · x=6: 1.89 misura (buco = vuoto) · x=7: 1.9 misura (buco = vuoto) · x=8: 1.72 misura (buco = vuoto) · x=9: 1.74 misura (buco = vuoto) · x=10: 1.74 misura (buco = vuoto) · x=11: 1.84 misura (buco = vuoto) · x=12: 2.09 misura (buco = vuoto) · x=13: 2.05 misura (buco = vuoto) · x=14: 2.17 misura (buco = vuoto) · x=15: 2.44 misura (buco = vuoto) · x=16: 2.43 misura (buco = vuoto) · x=17: 2.67 misura (buco = vuoto) · x=30: 4.47 misura (buco = vuoto) · x=31: 4.52 misura (buco = vuoto) · x=32: 4.5 misura (buco = vuoto) · x=33: 4.28 misura (buco = vuoto) · x=34: 4.17 misura (buco = vuoto) · x=35: 4.3 misura (buco = vuoto) · x=36: 4 misura (buco = vuoto) · x=37: 4.01 misura (buco = vuoto) · x=38: 3.92 misura (buco = vuoto) · x=39: 3.9 misura (buco = vuoto) · x=40: 3.83 misura (buco = vuoto) · x=41: 3.76 misura (buco = vuoto) · x=42: 3.78 misura (buco = vuoto) · x=43:

## 3 · Prevedere l'energia della prossima settimana
Sei settimane di kWh orari: cinque di addestramento, l'ultima come **prova a cui il modello non ha mai guardato**. Il confronto è con la MASE rispetto al seasonal naive settimanale ($m=168$): sotto 1 il modello batte il riferimento.

In [ ]:
var energia = File.ReadLines(Path.Combine(data, "energia_oraria.csv")).Skip(1).Select(l => l.Split(',')).ToArray();
double[] kwh = energia.Select(r => Num(r[1])).ToArray();
double[] train = kwh[..840], test = kwh[840..];

var candidates = new (string Name, double[] Forecast)[]
{
    ("sn24", ReferenceForecasts.SeasonalNaive<double>(train, 24, 168)),
    ("sn168", ReferenceForecasts.SeasonalNaive<double>(train, 168, 168)),
    ("hw24", ExponentialSmoothing.HoltWinters<double>(train, 24, 0.2, 0.01, 0.2, 168, 1.0).Forecast),
    ("hw168", ExponentialSmoothing.HoltWinters<double>(train, 168, 0.2, 0.01, 0.2, 168, 1.0).Forecast),
};
foreach (var (name, f) in candidates)
{
    P($"mase_{name}", Accuracy.Mase<double>(test, f, train, 168));
    P($"kwh_sett_{name}", f.Sum());
}
P("kwh_sett_reale", test.Sum());
foreach (var (name, f) in candidates) P($"kwh_errore_pct_{name}", 100 * (f.Sum() - test.Sum()) / test.Sum());
var best = candidates.OrderBy(c => Accuracy.Mase<double>(test, c.Forecast, train, 168)).First();
Console.WriteLine($"modello_scelto: {best.Name}");

mase_sn24: 1.870867
kwh_sett_sn24: 6478.850000
mase_sn168: 0.918371
kwh_sett_sn168: 7119.510000
mase_hw24: 1.903675
kwh_sett_hw24: 6360.663626
mase_hw168: 0.933787
kwh_sett_hw168: 7414.558830
kwh_sett_reale: 7395.980000
kwh_errore_pct_sn24: -12.400385
kwh_errore_pct_sn168: -3.738112
kwh_errore_pct_hw24: -13.998366
kwh_errore_pct_hw168: 0.251202
modello_scelto: sn168

In [ ]:
Chart.Lines("Ultima settimana: consumo reale e previsioni (kWh/h)",
    new Line("reale", test), new Line("seasonal naive 168", candidates[1].Forecast, Dashed: true), new Line("Holt-Winters 168", candidates[3].Forecast, Dashed: true));

Ultima settimana: consumo reale e previsioni (kWh/h) 20 30 40 50 60 70 0 50 100 150 200 <path d='M64 164.55L67.16 176.79L70.32 168.99L73.48 152.08L76.64 174.69L79.8 180.16L82.96 166.4L86.12 170.8L89.28 125.9L92.44 113.72L95.6 107.18L98.76 88.26L101.92 104.76L105.08 107.18L108.24 100.85L111.4 110.21L114.56 97.33L117.72 109.8L120.88 146.61L124.04 122.51L127.2 153.41L130.36 151.71L133.52 161.88L136.68 159.31L139.84 172.36L143 174.2L146.16 163.49L149.32 144.94L152.48 173.51L155.64 169.85L158.8 174.17L161.96 162.36L165.12 135.75L168.28 111.73L171.44 115.08L174.6 96.67L177.76 102.4L180.92 99.96L184.08 105.25L187.24 101.28L190.4 107.07L193.56 101.94L196.72 144.16L199.88 126.91L203.04 155.8L206.2 160L209.36 161.16L212.52 162.54L215.68 177.17L218.84 163.17L222 154.68L225.16 145.23L228.32 153.44L231.48 160.49L234.64 163.57L237.8 156L240.96 123.63L244.12 119.34L247.28 121.53L250.44 98.95L253.6 117.03L256.76 122.19L259.92 110.9L263.08 112.63L266.24 112.17L269.4 121.38L272.56 142.26L275.72 133.85L278.88 163.69L282.04 154.82L285.2 175.81L288.36 178.81L291.52 182.93L294.68 193.9L297.84 176.62L301 169.02L304.16 182.27L307.32 174.49L310.48 167.89L313.64 170.46L316.8 124L319.96 121.38L323.12 114.7L326.28 81.26L329.44 111.48L332.6 98.86L335.76 101.31L338.92 103.15L342.08 101.34L345.24 104.85L348.4 134.57L351.56 117.49L354.72 145.17L357.88 150.76L361.04 150.82L364.2 160L367.36 156.92L370.52 152.28L373.68 153.87L376.84 142.95L380 150.15L383.16 146.58L386.32 132.36L389.48 134.98L392.64 96.47L395.8 91.98L398.96 92.41L402.12 72.42L405.28 103.7L408.44 96.36L411.6 93.27L414.76 100.82L417.92 94.22L421.08 103.9L424.24 135.7L427.4 131.87L430.56 155.97L433.72 154.88L436.88 150.73L440.04 168.15L443.2 161.13L446.36 168.9L449.52 168.33L452.68 147.16L455.84 157.21L459 163.78L462.16 157.21L465.32 156.63L468.48 144.8L471.64 138.2L474.8 137.83L477.96 118.36L481.12 123.43L484.28 143.73L487.44 142.03L490.6 133.85L493.76 140.07L496.92 136.1L500.08 155.45L503.24 138.29L506.4 162.02L509.56 164.44L512.72 167.66L515.88 168.27L519.04 167.26L522.2 174.86L525.36 179.5L528.52 150.7L531.68 168.33L534.84 167.78L538 165.45L541.16 150.56L544.32 152.57L547.48 156.66L550.64 154.7L553.8 124.98L556.96 142.87L560.12 149.23L563.28 146.73L566.44 146.12L569.6 147.39L572.76 137.42L575.92 135.9L579.08 134.92L582.24 152.03L585.4 155.65L588.56 156.86L591.72 170.92' fill='none' stroke='#2a78d6' stroke-width='2' stroke-linejoin='round' stroke-linecap='round'/><path d='M64 167.35L67.16 172.1L70.32 183.88L73.48 161.33L76.64 173.8L79.8 165.04L82.96 175.38L86.12 172.47L89.28 128.55L92.44 133.91L95.6 127L98.76 95.84L101.92 116.11L105.08 110.15L108.24 113.69L111.4 110.96L114.56 112.51L117.72 115.65L120.88 162.36L124.04 129.76L127.2 154.62L130.36 166.57L133.52 162.83L136.68 155.94L139.84 173.74L143 181.34L146.16 165.19L149.32 158.56L152.48 164.29L155.64 160.49L158.8 158.19L161.96 162.08L165.12 136.91L168.28 134L171.44 112.4L174.6 98.83L177.76 110.06L180.92 108.22L184.08 97.25L187.24 97.42L190.4 100.13L193.56 99.18L196.72 144.28L199.88 116L203.04 147.24L206.2 162.91L209.36 158.94L212.52 156.26L215.68 169.08L218.84 173.11L222 171.35L225.16 159.54L228.32 176.16L231.48 174.17L234.64 161.41L237.8 162.34L240.96 120.03L244.12 121.79L247.28 116.8L250.44 88.44L253.6 110.04L256.76 107.64L259.92 103.04L263.08 106.75L266.24 105.48L269.4 107.36L272.56 150.93L275.72 138.26L278.88 155.42L282.04 158.88L285.2 160.41L288.36 168.53L291.52 163.66L294.68 175.7L297.84 174.37L301 162.31L304.16 182.12L307.32 175.56L310.48 180.19L313.64 172.13L316.8 142.41L319.96 134.57L323.12 128.27L326.28 108.97L329.44 119.05L332.6 102.86L335.76 103.15L338.92 111.01L342.08 104.97L345.24 113.49L348.4 144.83L351.56 131.64L354.72 152.49L357.88 154.59L361.04 160.18L364.2 159.54L367.36 173.63L370.52 173.05L373.68 174.32L376.84 152.2L380 173.68L383.16 164.55L386.32 158.04L389.48 159.72L392.64 119.11L395.8 111.79L398.96 105.77L402.12 86.59L405.28 94.6L408.44 101.11L411.6 103.27L414.76 103.01L417.92 104.68L421.08 113.29L424.24 144.65L427.4 129.4

> **Due criteri, due risposte.** Per il **profilo orario** (MASE) vince il seasonal naive settimanale; per il **totale della settimana** (il budget di energia) è Holt-Winters a sbagliare meno. E con **una sola settimana di prova** la scelta del modello e la sua valutazione coincidono: il risultato è ottimista. La validazione seria ripete la prova su molte origini (*rolling origin*, [infografica sulla validazione](../../../../refs/timeseries/)): è fra le teorie del libro non ancora implementate.

## 4 · Il compressore sta cedendo? La volatilità della vibrazione
La vibrazione dipende dallo stato del compressore (acceso/spento). Tolto il livello di ciascuno stato, restano shock la cui **varianza** dice quanto il compressore è "inquieto". Il GARCH si addestra sulle **prime 48 ore** (comportamento normale) e poi si legge la volatilità condizionale sulle 24 ore successive.

In [ ]:
var on = current.Select(c => c > 10).ToArray();
double MeanState(bool state) { double s = 0; var n = 0; for (var i = 0; i < 2880; i++) if (on[i] == state) { s += vib[i]; n++; } return s / n; }
var mOn = MeanState(true); var mOff = MeanState(false);
P("vib_livello_on_off", new[] { mOn, mOff });
var residual = vib.Select((v, i) => v - (on[i] ? mOn : mOff)).ToArray();

var garch = Garch11.FitVarianceTargeting<double>(residual[..2880]);
P("garch_fit", new[] { garch.Omega, garch.Alpha, garch.Beta });
var sigmaBar = Math.Sqrt(garch.UnconditionalVariance);
P("garch_sigma_base", sigmaBar);
var sigma = garch.ConditionalVariances(residual).Select(Math.Sqrt).ToArray();
var alarm = sigma.Select(s => s > 1.5 * sigmaBar).ToArray();
double Rate(int from, int to) { var n = 0; for (var i = from; i < to; i++) if (alarm[i]) n++; return (double)n / (to - from); }
P("allarme_giorno1", Rate(0, 1440));
P("allarme_giorno2", Rate(1440, 2880));
P("allarme_giorno3", Rate(2880, 4320));
// Allarme sostenuto: almeno 8 minuti sopra soglia nell'ultima ora (finestra mobile di 60 minuti).
int firstAlarm = -1, window = 0;
for (var i = 0; i < alarm.Length; i++)
{
    if (alarm[i]) window++;
    if (i >= 60 && alarm[i - 60]) window--;
    if (window >= 8 && firstAlarm < 0) firstAlarm = i;
}
P("allarme_primo_minuto", firstAlarm);
P("allarme_ritardo_min", firstAlarm - 2880);   // il cambiamento vero è al minuto 2880 (48 h): negativo = falso allarme

vib_livello_on_off: 2.599607 0.400610
garch_fit: 0.001890 0.082975 0.809350
garch_sigma_base: 0.132491
allarme_giorno1: 0.004861
allarme_giorno2: 0.002083
allarme_giorno3: 0.227083
allarme_primo_minuto: 2951.000000
allarme_ritardo_min: 71.000000

In [ ]:
Chart.Lines("Volatilità σ_t della vibrazione e soglia (1 punto ogni 15 min)",
    new Line("σ_t", Enumerable.Range(0, sigma.Length / 15).Select(i => sigma[i * 15]).ToArray()),
    new Line("soglia", Enumerable.Repeat(1.5 * sigmaBar, sigma.Length / 15).ToArray(), Dashed: true));

Volatilità σ_t della vibrazione e soglia (1 punto ogni 15 min) 0.1 0.2 0.3 0.4 0.5 0 100 200 300 <path d='M64 184.37L66.11 204.55L68.21 203.72L70.32 201.68L72.43 197.09L74.53 200.26L76.64 200.32L78.75 203.35L80.85 201.07L82.96 203.11L85.07 203.09L87.17 201.16L89.28 188.28L91.39 204.11L93.49 197.44L95.6 189.81L97.71 197.31L99.81 202.09L101.92 197.68L104.03 187.74L106.13 200.92L108.24 207.24L110.35 204L112.45 200.74L114.56 204.24L116.67 193.65L118.77 172.32L120.88 195.08L122.99 196.08L125.09 199.56L127.2 192.2L129.31 192.97L131.41 198.09L133.52 194.31L135.63 181.93L137.73 196.67L139.84 202.09L141.95 192.13L144.05 197.47L146.16 201.02L148.27 201.76L150.37 203.77L152.48 196.21L154.59 200.25L156.69 198.16L158.8 192.12L160.91 199.79L163.01 201.31L165.12 192.31L167.23 203.15L169.33 205.13L171.44 195.83L173.55 195.76L175.65 205.13L177.76 198.8L179.87 199.25L181.97 204.52L184.08 194.7L186.19 201.69L188.29 197.81L190.4 204.81L192.51 192.95L194.61 197.53L196.72 199.72L198.83 191.04L200.93 201.21L203.04 202.06L205.15 197.15L207.25 200.84L209.36 200.19L211.47 206.27L213.57 203.26L215.68 206.82L217.79 200.07L219.89 202.68L222 204.4L224.11 204.49L226.21 203.88L228.32 200.04L230.43 198.71L232.53 201.27L234.64 194.69L236.75 200.62L238.85 198.84L240.96 195.12L243.07 199.71L245.17 205.62L247.28 194.78L249.39 205.83L251.49 197.89L253.6 200.38L255.71 206.86L257.81 194.9L259.92 204.36L262.03 195.54L264.13 196.68L266.24 206.3L268.35 200.52L270.45 205.19L272.56 197.88L274.67 199.48L276.77 199.57L278.88 195.65L280.99 205.66L283.09 201.79L285.2 205.92L287.31 200.23L289.41 200.95L291.52 195.11L293.63 189.89L295.73 199.94L297.84 201.98L299.95 194.47L302.05 190.07L304.16 192.11L306.27 204L308.37 205.37L310.48 197L312.59 200.77L314.69 199.01L316.8 185.83L318.91 189.58L321.01 198.15L323.12 195.5L325.23 197.66L327.33 205.21L329.44 204.25L331.55 194.26L333.65 196.47L335.76 206.6L337.87 205.81L339.97 202.59L342.08 207.38L344.19 198.26L346.29 205.98L348.4 189.48L350.51 196.48L352.61 201.09L354.72 203.23L356.83 197.63L358.93 197.82L361.04 201.09L363.15 195.16L365.25 195.32L367.36 191.55L369.47 203.25L371.57 204.62L373.68 198.9L375.79 204.14L377.89 197.37L380 201.47L382.11 198.5L384.21 198.87L386.32 199L388.43 192.93L390.53 202.71L392.64 205.17L394.75 202.71L396.85 192.73L398.96 200.43L401.07 192.46L403.17 203.21L405.28 202.8L407.39 196.29L409.49 198.39L411.6 200.19L413.71 204.32L415.81 193.94L417.92 202.76L420.03 198.99L422.13 199.75L424.24 201.92L426.35 190.57L428.45 200.01L430.56 191.28L432.67 190.03L434.77 194.7L436.88 197.87L438.99 202.45L441.09 203.72L443.2 193.44L445.31 203.38L447.41 186.09L449.52 188.55L451.63 202.8L453.73 200.85L455.84 197.42L457.95 202.46L460.05 202.15L462.16 203.85L464.27 201.5L466.37 198.24L468.48 188L470.59 186.49L472.69 189.94L474.8 182.21L476.91 189.73L479.01 164.03L481.12 176.28L483.23 189.7L485.33 175.84L487.44 188.71L489.55 161.48L491.65 183.37L493.76 162.38L495.87 171.56L497.97 192.68L500.08 185.47L502.19 172.94L504.29 196.13L506.4 196.7L508.51 198.46L510.61 195.66L512.72 191.4L514.83 182.69L516.93 194.64L519.04 178.59L521.15 185.99L523.25 181.47L525.36 163.45L527.47 165.57L529.57 189.55L531.68 181.55L533.79 182.33L535.89 191.33L538 174.97L540.11 191.36L542.21 188.74L544.32 202.19L546.43 164.14L548.53 123.74L550.64 154.16L552.75 174.98L554.85 174.86L556.96 194.97L559.07 201.22L561.17 180.1L563.28 171.18L565.39 189.12L567.49 193.77L569.6 189.76L571.71 190.1L573.81 181.79L575.92 174.11L578.03 171.48L580.13 187.79L582.24 182.51L584.35 192.02L586.45 194.98L588.56 172.54L590.67 177.6L592.77 172.25L594.88 181.23L596.99 190.64L599.09 169.78L601.2 191.33L603.31 192.55L605.41 190.24L607.52 192.95L609.63 189.22L611.73 192.96L613.84 179.42L615.95 181.13L618.05 180.08L620.16 190.24L622.27 192.19L624.37 184.08L626.48 192.4L628.59 193.3L630.69 183.93L632.8 193.8L634.91 186.18L637.01 196.33L639.12 189.75L641.23 149.49L643.33 169.07L645.44 173.09L647.55 79.55L649.65 163.56L651.76 174.5L653.87 180.89L655.97 192.78L658.08 166.63L660.19 150.76L66

## 5 · Segnali accoppiati: VAR
Medie su 5 minuti di temperatura (filtrata), corrente e pressione. Il VAR(1) risponde: *quanto la corrente di adesso predice la temperatura fra cinque minuti?*

In [ ]:
const int step = 5;
var n5 = temp.Length / step;
var series = new double[n5, 3];
for (var k = 0; k < n5; k++)
{
    double st = 0, sc = 0, sp = 0;
    for (var j = 0; j < step; j++) { st += level[k * step + j]; sc += current[k * step + j]; sp += pressure[k * step + j]; }
    series[k, 0] = st / step; series[k, 1] = sc / step; series[k, 2] = sp / step;
}
var var3 = VarModel.Fit<double>(series, 1);
var a1 = var3.Coefficients[0];
P("var_a1", new[] { a1[0,0], a1[0,1], a1[0,2], a1[1,0], a1[1,1], a1[1,2], a1[2,0], a1[2,1], a1[2,2] });

var_a1: 0.841806 -0.032525 -0.068756 9.057558 0.728416 3.997443 -0.463599 -0.038371 -0.227008

Lettura di $A_1$ (righe = equazioni, colonne = temperatura, corrente, pressione al passo precedente): **temperatura ← corrente** è negativa (compressore acceso ⇒ la cella si raffredda); **corrente ← temperatura** è fortemente positiva (cella calda ⇒ il termostato accende il compressore). Il VAR ritrova, dai soli dati, la logica del termostato.

## 6 · Comunicare: KPI, dashboard e risposta al cliente

In [ ]:
var minutes72 = temp.Length;
var energyNext = best.Forecast.Sum();
var alarmDay3 = Rate(2880, 4320);

Kpi.Tiles(
    new Tile("Tempo sopra +4 °C", $"{observed}–{observed + suspect.Sum(g => g.Length)} min", "minimo osservato – massimo non escludibile", "warn"),
    new Tile("Escursioni HACCP", $"{Runs(level)}", $"temperatura massima stimata {level.Max():F1} °C", "warn"),
    new Tile("Dati mancanti", $"{100.0 * gaps.Sum(g => g.Length) / minutes72:F1} %", $"{gaps.Count} buchi, il più lungo {gaps.Max(g => g.Length)} min", "ok"),
    new Tile("Energia prossima settimana", $"{energyNext:F0} kWh", $"modello {best.Name}, MASE {Accuracy.Mase<double>(test, best.Forecast, train, 168):F2}; sul totale {100 * (energyNext - test.Sum()) / test.Sum():+0.0;-0.0} % nella settimana di prova", "ok"),
    new Tile("Vibrazione compressore", $"{100 * alarmDay3:F0} % allarme", $"ultime 24 h, contro {100 * Rate(0, 1440):F1} % il primo giorno; primo allarme {firstAlarm - 2880} min dopo il cambio", alarmDay3 > 0.2 ? "alarm" : "ok"));

Tempo sopra +4 °C 7–19 min minimo osservato – massimo non escludibile ▲ attenzione Escursioni HACCP 1 temperatura massima stimata 4.5 °C ▲ attenzione Dati mancanti 1.6 % 6 buchi, il più lungo 20 min ✓ nella norma Energia prossima settimana 7120 kWh modello sn168, MASE 0.92; sul totale -3.7 % nella settimana di prova ✓ nella norma Vibrazione compressore 23 % allarme ultime 24 h, contro 0.5 % il primo giorno; primo allarme 71 min dopo il cambio ✖ allarme

In [ ]:
var answer = $"Nelle 72 ore dal 14 al 16 settembre la temperatura della cella è stata osservata sopra +4 °C per almeno {observed} minuti, in {Runs(level)} escursione, con un massimo stimato di {level.Max():F1} °C (martedì pomeriggio). " +
             $"In {suspect.Sum(g => g.Length)} minuti il sensore non ha comunicato e la temperatura ai bordi del buco differiva di oltre 1 °C: non possiamo escludere altri minuti sopra soglia, per un massimo di {observed + suspect.Sum(g => g.Length)}. " +
             $"Per questo la nostra risposta non è «sempre sotto +4 °C», ma «da {observed} a {observed + suspect.Sum(g => g.Length)} minuti sopra soglia in un'unica escursione»; proponiamo di verificare la porta della cella e il gateway di comunicazione.";
Kpi.Quote("Risposta al cliente: «la mia merce è stata sempre sotto +4 °C?»", answer);

var maintenance = $"La volatilità della vibrazione del compressore è oltre soglia nel {100 * alarmDay3:F0} % dell'ultimo giorno (contro il {100 * Rate(0, 1440):F1} % del primo). L'allarme sostenuto è scattato {firstAlarm - 2880} minuti dopo l'inizio del cambiamento: consigliato un controllo del cuscinetto.";
Kpi.Quote("Nota per la manutenzione", maintenance);

Risposta al cliente: «la mia merce è stata sempre sotto +4 °C?» Nelle 72 ore dal 14 al 16 settembre la temperatura della cella è stata osservata sopra +4 °C per almeno 7 minuti, in 1 escursione, con un massimo stimato di 4.5 °C (martedì pomeriggio). In 12 minuti il sensore non ha comunicato e la temperatura ai bordi del buco differiva di oltre 1 °C: non possiamo escludere altri minuti sopra soglia, per un massimo di 19. Per questo la nostra risposta non è «sempre sotto +4 °C», ma «da 7 a 19 minuti sopra soglia in un'unica escursione»; proponiamo di verificare la porta della cella e il gateway di comunicazione.

Nota per la manutenzione La volatilità della vibrazione del compressore è oltre soglia nel 23 % dell'ultimo giorno (contro il 0.5 % del primo). L'allarme sostenuto è scattato 71 minuti dopo l'inizio del cambiamento: consigliato un controllo del cuscinetto.

## Cosa abbiamo imparato
- **Un buco nei dati è un'informazione**: il KPI HACCP si dà come intervallo, e la verità nota (11 minuti) cade dentro.
- **Il Kalman fa il suo lavoro ma non indovina**: non vede una salita avvenuta interamente nel buco, né con il solo livello né con il trend.
- **MASE batte la percentuale**: i kWh non sono mai zero, ma il metodo è lo stesso per la produzione fotovoltaica, che di notte vale zero (scenario D).
- **GARCH come campanello**: non dice *cosa* sta cedendo, dice che il comportamento è cambiato; l'allarme non coincide con il cambiamento (*change time is not alarm time*).
- **Il PySharp dà gli stessi numeri**: stessa libreria, stessa risposta, linguaggio a scelta di chi lavora sull'impianto.